# FANet: Comprehensive Benchmark
**Architecture:** ResNet-34 Encoder + **MFAD**
**Loss Function:** BCE + Dice Loss
**Data Pipeline:** Dynamic Intensity Normalization + Exact Train/Test Split Ratios matching the Original Paper


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from tqdm import tqdm
from PIL import Image
import tifffile as tiff
import albumentations as A

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models


In [ ]:
# ==============================================================================
# CẤU HÌNH DATASET THEO ĐÚNG CHUẨN PAPER (TỪ benchmark.png)
# ==============================================================================
DATASET_NAME = "Kvasir-SEG"
DATA_DIR = "/kaggle/input/datasets/namnguynnnn/kvasir-seg"
IMG_SUBDIR = "images"
MASK_SUBDIR = "masks"

N_TRAIN = 880
N_TEST = 120
IMAGE_SIZE = (256, 256)
EPOCHS = 150 
BATCH_SIZE = 8

# ==============================================================================
# BƯỚC 2: CẤU HÌNH TRAINING
# ==============================================================================
VAL_BATCH_SIZE = 1 # Chuẩn hoá Dataset-level Mean Dice
LR = 1e-4
import torch
import os
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Checking dataset: {DATASET_NAME} at {DATA_DIR}...")
if not os.path.exists(os.path.join(DATA_DIR, IMG_SUBDIR)) and 'electron-microscopy' not in DATA_DIR.lower():
    for root, dirs, files in os.walk(DATA_DIR):
        dirs_lower = [d.lower() for d in dirs]
        if any(x in dirs_lower for x in ['images', 'original', 'input', 'training']) and \
           any(x in dirs_lower for x in ['masks', 'ground truth', 'labels', 'manual', '1st_manual']):
            IMG_SUBDIR = dirs[dirs_lower.index(next(x for x in ['images', 'original', 'input', 'training'] if x in dirs_lower))]
            MASK_SUBDIR = dirs[dirs_lower.index(next(x for x in ['masks', 'ground truth', 'labels', 'manual', '1st_manual'] if x in dirs_lower))]
            DATA_DIR = root
            break

print(f"✅ FINAL PATHS: Images -> {os.path.join(DATA_DIR, IMG_SUBDIR)} | Masks -> {os.path.join(DATA_DIR, MASK_SUBDIR)}")


In [ ]:
class UniversalDataset(Dataset):
    def __init__(self, images_path, masks_path, is_train=False):
        self.images_path = images_path
        self.masks_path = masks_path
        self.n_samples = len(images_path)
        self.is_train = is_train
        
        # Heavy Albumentations Data Augmentation để tái tạo x32 Augmented Train Data của paper
        self.transform = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=45, 
                               interpolation=cv2.INTER_LINEAR, 
                               mask_interpolation=cv2.INTER_NEAREST, p=0.6),
            A.GridDistortion(p=0.3),
            A.ElasticTransform(p=0.3, alpha=120, sigma=120 * 0.05, alpha_affine=120 * 0.03)
        ])
        
        self.is_3d_volume = False
        if self.n_samples == 1 and str(images_path[0]).endswith('.tif'):
            img_vol = tiff.imread(images_path[0])
            if len(img_vol.shape) == 3:
                self.is_3d_volume = True
                self.img_vol = img_vol
                self.mask_vol = tiff.imread(masks_path[0])
                self.n_samples = self.img_vol.shape[0]

    def __getitem__(self, index):
        if self.is_3d_volume:
            image = self.img_vol[index]
            mask = self.mask_vol[index]
            if len(image.shape) == 2:
                image = np.stack([image]*3, axis=-1)
        else:
            img_path = self.images_path[index]
            mask_path = self.masks_path[index]
            
            # --- XỬ LÝ ĐỌC ẢNH CHUẨN XÁC ---
            if img_path.lower().endswith('.gif'):
                image = np.array(Image.open(img_path).convert('RGB'))
            elif img_path.lower().endswith('.tif') or img_path.lower().endswith('.tiff'):
                image = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
                if image is None: # Fallback nếu cv2 lỗi
                    image = np.array(Image.open(img_path).convert('RGB'))
                else:
                    if len(image.shape) == 2:
                        image = np.stack([image]*3, axis=-1)
                    elif len(image.shape) == 3 and image.shape[2] == 3:
                        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                    elif len(image.shape) == 3 and image.shape[2] == 4:
                        image = cv2.cvtColor(image, cv2.COLOR_BGRA2RGB)
            else:
                image = np.array(Image.open(img_path).convert('RGB'))
                
            # Đọc mask
            if mask_path.lower().endswith('.gif'):
                mask = np.array(Image.open(mask_path).convert('L'))
            elif mask_path.lower().endswith('.tif') or mask_path.lower().endswith('.tiff'):
                mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
                if mask is None: # Fallback nếu cv2 lỗi
                    mask = np.array(Image.open(mask_path).convert('L'))
                elif len(mask.shape) == 3:
                    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
            else:
                mask = np.array(Image.open(mask_path).convert('L'))

        # Resize
        image = cv2.resize(image, IMAGE_SIZE, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, IMAGE_SIZE, interpolation=cv2.INTER_NEAREST)

        # Cường hóa Augmentation nếu là tập Train
        if self.is_train and not self.is_3d_volume:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        # --- DYNAMIC INTENSITY NORMALIZATION (FIX THE 127 THRESHOLD BUG) ---
        image = image.astype(np.float32)
        img_max = image.max()
        if img_max > 0:
            if img_max > 255:
                image = image / 65535.0
            else:
                image = image / 255.0

        image = np.transpose(image, (2, 0, 1))
        
        # Binarize Mask Tự động dựa trên np.unique
        mask = mask.astype(np.float32)
        unique_vals = np.unique(mask)
        if len(unique_vals) > 2:
            thresh = (mask.max() + mask.min()) / 2.0
            mask = (mask > thresh).astype(np.float32)
        else:
            if mask.max() > 0:
                mask = (mask == mask.max()).astype(np.float32)

        mask = np.expand_dims(mask, axis=0)

        return image, mask

    def __len__(self):
        return self.n_samples

# ==============================================================================
# BƯỚC 3: PHÂN CHIA DỮ LIỆU ĐÚNG RATIO NHƯ BẢNG BENCHMARK CỦA PAPER
# ==============================================================================
exts = ['*.jpg', '*.png', '*.tif', '*.tiff', '*.jpeg', '*.gif']
images, masks = [], []

if "electron-microscopy-3d-segmentation" in DATA_DIR.lower():
    images = [os.path.join(DATA_DIR, "training.tif")]
    masks = [os.path.join(DATA_DIR, "training_groundtruth.tif")]
elif DATASET_NAME == "DRIVE":
    for ext in exts:
        images.extend(glob(os.path.join(DATA_DIR, '**', 'images', '**', ext), recursive=True))
        masks.extend(glob(os.path.join(DATA_DIR, '**', '1st_manual', '**', ext), recursive=True))
else:
    for ext in exts:
        images.extend(glob(os.path.join(DATA_DIR, IMG_SUBDIR, '**', ext), recursive=True))
        masks.extend(glob(os.path.join(DATA_DIR, MASK_SUBDIR, '**', ext), recursive=True))

images = sorted(images)
masks = sorted(masks)

if len(images) == 0:
    raise ValueError("❌ DATASET EMPTY! Could not find images.")
if len(images) != len(masks):
    print(f"⚠️ Mismatch: {len(images)} images vs {len(masks)} masks. Truncating to minimum.")
    min_len = min(len(images), len(masks))
    images, masks = images[:min_len], masks[:min_len]

# CHIA SPLIT THEO ĐÚNG SỐ LƯỢNG (N_TRAIN, N_TEST) TỪ PAPER
if "electron-microscopy-3d-segmentation" in DATA_DIR.lower():
    train_x, train_y = images, masks
    valid_x = [os.path.join(DATA_DIR, "testing.tif")]
    valid_y = [os.path.join(DATA_DIR, "testing_groundtruth.tif")]
else:
    import random
    combined = list(zip(images, masks))
    random.seed(42)
    random.shuffle(combined)
    images, masks = zip(*combined)
    
    total_imgs = len(images)
    if total_imgs >= N_TRAIN + N_TEST:
        actual_train = N_TRAIN
        actual_test = N_TEST
    else:
        print(f"⚠️ Found only {total_imgs} images (Expected {N_TRAIN + N_TEST}). Using 80:20 fallback split.")
        actual_test = max(1, int(total_imgs * 0.2))
        actual_train = total_imgs - actual_test
    
    train_x = images[:actual_train]
    train_y = masks[:actual_train]
    
    valid_x = images[-actual_test:]
    valid_y = masks[-actual_test:]

train_dataset = UniversalDataset(train_x, train_y, is_train=True)
valid_dataset = UniversalDataset(valid_x, valid_y, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=VAL_BATCH_SIZE, shuffle=False)

print(f'✅ {DATASET_NAME} loaded! Paper split expected: Train {N_TRAIN} | Test {N_TEST}')
print(f'   -> Actual split loaded: Train size: {len(train_dataset)} | Valid size: {len(valid_dataset)}')


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))))

class FANet_MFAD(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        
        self.e1_conv = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            self.e1_conv.weight[:, :3, :, :] = resnet.conv1.weight
            self.e1_conv.weight[:, 3:4, :, :] = 0.0
        self.e1_bn = resnet.bn1
        self.e1_relu = resnet.relu
        self.pool = resnet.maxpool
        
        self.e2 = resnet.layer1
        self.e3 = resnet.layer2
        self.e4 = resnet.layer3
        self.b  = resnet.layer4
        
        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.d1 = ConvBlock(256 + 256, 256)
        self.att1 = nn.Sequential(nn.Conv2d(1, 256, 1), nn.BatchNorm2d(256), nn.Sigmoid())
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.d2 = ConvBlock(128 + 128, 128)
        self.att2 = nn.Sequential(nn.Conv2d(1, 128, 1), nn.BatchNorm2d(128), nn.Sigmoid())
        
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.d3 = ConvBlock(64 + 64, 64)
        self.att3 = nn.Sequential(nn.Conv2d(1, 64, 1), nn.BatchNorm2d(64), nn.Sigmoid())
        
        self.up4 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.d4 = ConvBlock(64 + 64, 64)
        self.att4 = nn.Sequential(nn.Conv2d(1, 64, 1), nn.BatchNorm2d(64), nn.Sigmoid())
        
        self.up5 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.d5 = ConvBlock(32, 32)
        self.att5 = nn.Sequential(nn.Conv2d(1, 32, 1), nn.BatchNorm2d(32), nn.Sigmoid())
        
        self.out = nn.Conv2d(32, 1, kernel_size=1)
        self.learned_gate = nn.Conv2d(1, 1, kernel_size=1)

    def apply_mfad(self, feat, prev_mask, att_layer):
        scaled_mask = F.interpolate(prev_mask, size=feat.shape[2:], mode='nearest')
        att = att_layer(scaled_mask)
        return feat * (1 + att)

    def forward(self, x, prev_mask=None):
        if prev_mask is None:
            prev_mask = torch.zeros(x.size(0), 1, x.size(2), x.size(3)).to(x.device)
        else:
            prev_mask = torch.sigmoid(self.learned_gate(prev_mask.detach()))
            
        x_in = torch.cat([x, prev_mask], dim=1)
        s1 = self.e1_relu(self.e1_bn(self.e1_conv(x_in)))
        s2 = self.e2(self.pool(s1))
        s3 = self.e3(s2)
        s4 = self.e4(s3)
        b = self.b(s4)
        
        d1 = self.d1(torch.cat([self.up1(b), s4], dim=1))
        d1 = self.apply_mfad(d1, prev_mask, self.att1)
        
        d2 = self.d2(torch.cat([self.up2(d1), s3], dim=1))
        d2 = self.apply_mfad(d2, prev_mask, self.att2)
        
        d3 = self.d3(torch.cat([self.up3(d2), s2], dim=1))
        d3 = self.apply_mfad(d3, prev_mask, self.att3)
        
        d4 = self.d4(torch.cat([self.up4(d3), s1], dim=1))
        d4 = self.apply_mfad(d4, prev_mask, self.att4)
        
        d5 = self.d5(self.up5(d4))
        d5 = self.apply_mfad(d5, prev_mask, self.att5)
        
        return self.out(d5)


In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, inputs, targets, smooth=1):
        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        bce = F.binary_cross_entropy(inputs, targets, reduction='mean')
        intersection = (inputs * targets).sum()                            
        dice = (2.*intersection + smooth)/(inputs.sum() + targets.sum() + smooth)  
        return bce + (1 - dice)

def calculate_metrics(y_true, y_pred):
    y_true = y_true.detach().cpu().numpy()
    y_pred = (torch.sigmoid(y_pred).detach().cpu().numpy() > 0.5).astype(np.float32)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    dice = (2 * tp) / (2 * tp + fp + fn + 1e-6)
    prec = tp / (tp + fp + 1e-6)
    rec = tp / (tp + fn + 1e-6)
    fpr = fp / (fp + tn + 1e-6)
    return dice, prec, rec, fpr


In [ ]:
model = FANet_MFAD().to(DEVICE)
criterion = DiceBCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

ckpt_name = f"fanet_{DATASET_NAME.replace(' ', '_')}_mfad_best.pth"

print(f"Starting Training on {DATASET_NAME}...")
best_dice = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for x, y in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]'):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        out = model(x, torch.sigmoid(out))
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    model.eval()
    val_loss, val_dice = 0, 0
    with torch.no_grad():
        for x, y in tqdm(valid_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Valid]'):
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            out = model(x, torch.sigmoid(out))
            loss = criterion(out, y)
            val_loss += loss.item()
            dice, _, _, _ = calculate_metrics(y, out)
            val_dice += dice
            
    train_loss /= len(train_loader)
    val_loss /= len(valid_loader)
    val_dice /= len(valid_loader)
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1} | LR: {current_lr:.6f} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f}")
    
    scheduler.step()
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), ckpt_name)
